# LughaGen evaluation on Google Colab

Runs Belebele, IrokoBench (AfriMMLU / AfriXNLI / AfriMGSM), FLORES-200 and SIB-200 on models stored in **Google Drive**.

**How a run works.** One run is split across as many Colab sessions as it needs. Each session stops itself cleanly after `MAX_RUNTIME_HOURS`, saves everything to Drive, and the next session carries on exactly where it stopped. So the routine is simply: **Runtime ▸ Run all**, wait, repeat until the status cell says everything is complete.

**One-time setup**
1. Runtime ▸ Change runtime type ▸ **GPU**.
2. Put your model folders on Drive and copy `notebooks/models.example.json` to `MyDrive/lughagen/models.json`, fixing the paths.
3. Colab left sidebar ▸ 🔑 **Secrets**, add (and switch on *Notebook access*):
   - `GITHUB_TOKEN`: a fine-grained personal access token limited to this one repository, with *Contents: Read and write*.
   - `HF_TOKEN`: a Hugging Face token (needed for the gated FLORES dataset; accept its terms on the dataset page first).

**Exit codes of the run cell:** `0` all finished · `10` time budget reached (run again) · `20` aborted, e.g. GPU out of memory (restart the runtime, run again) · `30` finished but some examples failed (run again to retry only those).

## 1. Check the GPU
Free Colab does not guarantee a particular GPU, so look at what this session got.

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "No GPU found: use Runtime > Change runtime type > GPU")

## 2. Settings
The only cell you normally edit.

In [ ]:
GITHUB_REPO = "https://github.com/K-Musembi/llm_evaluation.git"
BRANCH = "main"

DRIVE_ROOT = "/content/drive/MyDrive/lughagen"       # models.json and the saved results live here
MODELS_FILE = f"{DRIVE_ROOT}/models.json"
RUN_NAME = "lughagen_round2"                          # results folder name (Drive and GitHub)

# Recommended order: comprehension and knowledge first, translation and classification last.
TASKS = ["belebele", "afrimmlu", "afrixnli", "afrimgsm", "flores", "sib"]
MODELS = None            # None = every model in models.json, or e.g. ["Llama-8B-FFT"]
LANGUAGES = ["swh_Latn", "luo_Latn", "kik_Latn", "kam_Latn"]

QUANTIZATION = "4bit"    # "none" | "8bit" | "4bit". A 15 GB T4 needs 4bit (or 8bit) for an 8B model.
MAX_RUNTIME_HOURS = 5.5  # stop cleanly before Colab's own limits; the next session resumes
LIMIT = None             # e.g. 5 for a quick smoke test of the whole pipeline

REPO_DIR = "/content/llm_evaluation"
OUTPUT_DIR = f"/content/results/{RUN_NAME}"            # fast local disk
SYNC_DIR = f"{DRIVE_ROOT}/results/{RUN_NAME}"          # copied to Drive during the run, restored next session

## 3. Connect Drive, load secrets, install packages

In [ ]:
import os
import sys
import subprocess

from google.colab import drive, userdata

drive.mount("/content/drive")

for secret in ("GITHUB_TOKEN", "HF_TOKEN"):
    try:
        os.environ[secret] = userdata.get(secret)
        print(f"{secret}: loaded")
    except Exception:
        print(f"{secret}: NOT set (add it under Secrets and enable Notebook access)")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers", "accelerate", "bitsandbytes",
                "peft", "sacrebleu", "datasets", "sentencepiece"], check=True)
print("packages ready")

## 4. Get the code from GitHub
The token is used for this one command only and is not left in the repo's git config.

In [ ]:
import pathlib

token = os.environ.get("GITHUB_TOKEN")
auth_url = GITHUB_REPO.replace("https://", f"https://x-access-token:{token}@") if token else GITHUB_REPO

if pathlib.Path(REPO_DIR, ".git").exists():
    result = subprocess.run(["git", "-C", REPO_DIR, "pull", "--rebase", auth_url, BRANCH], capture_output=True, text=True)
else:
    result = subprocess.run(["git", "clone", "--branch", BRANCH, auth_url, REPO_DIR], capture_output=True, text=True)
    subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", GITHUB_REPO])

output = result.stdout + result.stderr
print(output.replace(token, "***") if token else output)
if result.returncode != 0:
    raise SystemExit("Could not fetch the repository (check GITHUB_TOKEN and the repo name).")

## 5. Helpers

In [ ]:
def run_script(script, *args):
    """Run a script from the repo, streaming its output live. Returns the exit code."""
    command = [sys.executable, "-u", f"{REPO_DIR}/scripts/{script}", *map(str, args)]
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                               cwd=f"{REPO_DIR}/scripts")
    for line in process.stdout:
        print(line, end="")
    return process.wait()


def eval_args():
    args = ["--backend", "local", "--model-paths-file", MODELS_FILE, "--quantization", QUANTIZATION,
            "--tasks", *TASKS, "--languages", *LANGUAGES,
            "--output-dir", OUTPUT_DIR, "--sync-dir", SYNC_DIR, "--cache-dir", "/content/hf_cache"]
    if MODELS:
        args += ["--models", *MODELS]
    return args

## 6. Check the setup (no evaluation yet)
Confirms every model folder exists on Drive and estimates whether it fits this GPU. Fix anything marked `PROBLEM` first.

In [ ]:
run_script("run_eval.py", *eval_args(), "--check-setup")

## 7. Progress so far
Shows which jobs are complete, partial, or not started (restores earlier sessions' results from Drive first).

In [ ]:
run_script("run_eval.py", *eval_args(), "--status")

## 8. Run the evaluation
Stops itself after `MAX_RUNTIME_HOURS`. Keep this tab open while it runs. If the session dies anyway, nothing is lost beyond the last few minutes: just run all again.

In [ ]:
extra = ["--max-runtime-hours", MAX_RUNTIME_HOURS]
if LIMIT:
    extra += ["--limit", LIMIT]

exit_code = run_script("run_eval.py", *eval_args(), *extra)

MEANINGS = {0: "All requested jobs finished.",
            10: "Time budget reached. Results are saved. Run all again in a new session to continue.",
            20: "Aborted (backend kept failing, e.g. GPU out of memory). Restart the runtime, then run all again.",
            30: "Finished, but some examples failed. Run all again to retry just those."}
print(f"\nexit code {exit_code}: {MEANINGS.get(exit_code, 'unexpected, read the output above')}")

## 9. Push results to GitHub
Publishes the small files (`summary.csv`, `dashboard.html`, per-job metrics and review CSVs) to `results/<RUN_NAME>/`. The big raw logs stay safely on Drive. Run this after each session, or just once at the end.

In [ ]:
run_script("publish_results.py", "--results-dir", OUTPUT_DIR, "--repo-dir", REPO_DIR, "--run-name", RUN_NAME, "--dry-run")

In [ ]:
# Happy with the list above? Publish for real.
run_script("publish_results.py", "--results-dir", OUTPUT_DIR, "--repo-dir", REPO_DIR, "--run-name", RUN_NAME)

## Next session
1. Runtime ▸ Run all (cells 6 and 7 tell you where things stand; cell 8 resumes automatically).
2. Repeat until cell 7 reports every job as `complete`, then push (cell 9).

To look at results: open `dashboard.html` from `results/<RUN_NAME>/` in the repo, or `summary.csv` for the raw numbers.